In [3]:
import torch
from torch import nn
from torch.nn import functional as F

net  = nn.Sequential(nn.Linear(4,8),nn.ReLU(),nn.Linear(8,1))
X = torch.rand(2,4)
net(X)

tensor([[0.4077],
        [0.4001]], grad_fn=<AddmmBackward0>)

In [2]:
print(net[2].state_dict())

OrderedDict({'weight': tensor([[ 0.1336,  0.0204,  0.1798,  0.1422,  0.0189,  0.1755, -0.1736,  0.3134]]), 'bias': tensor([0.0495])})


In [3]:
print(type(net[2].bias))

<class 'torch.nn.parameter.Parameter'>


In [5]:
print(net[2].bias)

Parameter containing:
tensor([0.0495], requires_grad=True)


In [6]:
print(net[2].bias.data)

tensor([0.0495])


In [8]:
print(net[2].weight.grad)

None


In [9]:
print(*[(name,param.shape) for name,param in net[0].named_parameters()])
print(*[(name,param.shape) for name,param in net.named_parameters()])

('weight', torch.Size([8, 4])) ('bias', torch.Size([8]))
('0.weight', torch.Size([8, 4])) ('0.bias', torch.Size([8])) ('2.weight', torch.Size([1, 8])) ('2.bias', torch.Size([1]))


In [15]:
net.state_dict()['2.bias'].data

tensor([0.0495])

In [14]:
net[2].state_dict()['bias'].data

tensor([0.0495])

In [16]:
def block1():
    return nn.Sequential(nn.Linear(4,8),nn.ReLU(),nn.Linear(8,4),nn.ReLU())

def block2():
    net = nn.Sequential()
    for i in range(4):
        # 在这里嵌套
        net.add_module(f'block{i}',block1())
    return net

rgnet = nn.Sequential(block2(),nn.Linear(4,1))
rgnet(X)

tensor([[-0.3450],
        [-0.3453]], grad_fn=<AddmmBackward0>)

In [17]:
print(rgnet)

Sequential(
  (0): Sequential(
    (block0): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block1): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block2): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block3): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
  )
  (1): Linear(in_features=4, out_features=1, bias=True)
)


In [19]:
rgnet[0][1][0].bias.data

tensor([ 0.3468,  0.2525, -0.2729, -0.3103, -0.1910, -0.4457, -0.1047,  0.4775])

In [7]:
def init_normal(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight,mean=0,std=0.01)
        nn.init.zeros_(m.bias)

net.apply(init_normal)
net[0].weight.data[0],net[0].bias.data[0]

(tensor([0.0115, 0.0155, 0.0123, 0.0059]), tensor(0.))

In [8]:
net[0].weight.data,net[0].bias.data

(tensor([[ 1.1458e-02,  1.5494e-02,  1.2272e-02,  5.9419e-03],
         [ 5.5716e-04,  2.0711e-03, -6.3374e-03, -4.8538e-03],
         [-4.9526e-03, -8.3914e-05, -2.2752e-03,  1.1552e-02],
         [ 3.8966e-03, -1.1564e-02,  2.2960e-02, -1.0591e-02],
         [ 1.9878e-02, -2.2567e-03, -9.3954e-03,  1.9328e-02],
         [-2.0286e-02, -1.7747e-02,  5.8641e-03,  1.1134e-02],
         [-2.0103e-03,  4.0945e-03, -1.3698e-02,  1.6099e-02],
         [-2.2673e-03, -8.0906e-03, -9.2767e-03, -3.7047e-03]]),
 tensor([0., 0., 0., 0., 0., 0., 0., 0.]))

In [5]:
def init_constant(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight,1)
        nn.init.zeros_(m.bias)

net.apply(init_constant)
net[0].weight.data[0],net[0].bias.data[0]

(tensor([1., 1., 1., 1.]), tensor(0.))

In [9]:
net[0].weight.data,net[0].bias.data

(tensor([[ 1.1458e-02,  1.5494e-02,  1.2272e-02,  5.9419e-03],
         [ 5.5716e-04,  2.0711e-03, -6.3374e-03, -4.8538e-03],
         [-4.9526e-03, -8.3914e-05, -2.2752e-03,  1.1552e-02],
         [ 3.8966e-03, -1.1564e-02,  2.2960e-02, -1.0591e-02],
         [ 1.9878e-02, -2.2567e-03, -9.3954e-03,  1.9328e-02],
         [-2.0286e-02, -1.7747e-02,  5.8641e-03,  1.1134e-02],
         [-2.0103e-03,  4.0945e-03, -1.3698e-02,  1.6099e-02],
         [-2.2673e-03, -8.0906e-03, -9.2767e-03, -3.7047e-03]]),
 tensor([0., 0., 0., 0., 0., 0., 0., 0.]))

In [12]:
def init_xavier(m):
    if type(m) == nn.Linear:
        nn.init.xavier_uniform_(m.weight)

def init_42(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight,42)

net[0].apply(init_xavier)
net[2].apply(init_42)
print(net[0].weight.data[0])
print(net[2].weight.data)

tensor([-0.4749,  0.6062,  0.1890, -0.3016])
tensor([[42., 42., 42., 42., 42., 42., 42., 42.]])


In [13]:
def my_init(m):
    if type(m) == nn.Linear:
        print("Init",*[(name,param.shape) for name,param in m.named_parameters()][0])
        nn.init.uniform_(m.weight,-10,10)
        m.weight.data *= m.weight.data.abs() >= 5

net.apply(my_init)
net[0].weight[:2]

Init weight torch.Size([8, 4])
Init weight torch.Size([1, 8])


tensor([[-9.8080, -9.4781,  0.0000, -8.3331],
        [-0.0000, -7.2847,  0.0000,  0.0000]], grad_fn=<SliceBackward0>)

In [15]:
net[0].weight.data[:]+=1
net[0].weight.data[0,0]=42
net[0].weight.data[0]

tensor([42.0000, -8.4781,  1.0000, -7.3331])

In [16]:
# 我们需要给共享层提供一个名称，以便可以引用它的参数
shared = nn.Linear(8,8)
net = nn.Sequential(nn.Linear(4,8),nn.ReLU(),
                    shared,nn.ReLU(),
                    shared,nn.ReLU(),
                    nn.Linear(8,1))
net(X)
# 检查参数是否相同
print(net[2].weight.data[0] == net[4].weight.data[0])
net[2].weight.data[0,0] = 100
# 确保它们实际上是同一个对象而不只是有相同的值
print(net[2].weight.data[0] == net[4].weight.data[0])

tensor([True, True, True, True, True, True, True, True])
tensor([True, True, True, True, True, True, True, True])
